In [1]:
!pip install ultralytics opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.0 MB/s eta 0:00:00


In [2]:
import cv2
import numpy as np
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
model = YOLO("yolov8n.pt")

In [4]:
from google.colab import files

uploaded = files.upload()

Saving input.mp4 to input.mp4


In [5]:
video = cv2.VideoCapture("input.mp4")

width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = video.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    "output.mp4",
    fourcc,
    fps,
    (width,height)
)

while video.isOpened():

    ret, frame = video.read()

    if not ret:
        break

    results = model(frame)

    command = "FORWARD"

    for result in results:

        boxes = result.boxes

        for box in boxes:

            x1,y1,x2,y2 = map(int, box.xyxy[0])

            confidence = float(box.conf[0])

            if confidence < 0.40:
                continue

            area = (x2-x1)*(y2-y1)

            center = (x1+x2)//2

            cv2.rectangle(
                frame,
                (x1,y1),
                (x2,y2),
                (0,255,0),
                2
            )

            if area > 50000:

                if center < width//3:

                    command = "MOVE RIGHT"

                elif center > 2*width//3:

                    command = "MOVE LEFT"

                else:

                    command = "STOP"

    cv2.putText(
        frame,
        command,
        (30,50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (0,0,255),
        3
    )

    out.write(frame)

video.release()
out.release()

print("Completed")


0: 384x640 2 persons, 376.6ms
Speed: 20.9ms preprocess, 376.6ms inference, 36.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 229.3ms
Speed: 13.7ms preprocess, 229.3ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 268.1ms
Speed: 5.4ms preprocess, 268.1ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 216.6ms
Speed: 6.9ms preprocess, 216.6ms inference, 4.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 205.2ms
Speed: 4.2ms preprocess, 205.2ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 155.5ms
Speed: 4.1ms preprocess, 155.5ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 135.8ms
Speed: 4.6ms preprocess, 135.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 137.8ms
Speed: 3.8ms preprocess, 137.8ms inference, 1.0ms postprocess per

In [6]:
from google.colab import files

files.download("output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import pandas as pd

video = cv2.VideoCapture("input.mp4")

commands = []

frame_no = 0

while video.isOpened():

    ret, frame = video.read()

    if not ret:
        break

    results = model(frame)

    command = "FORWARD"

    h,w,_ = frame.shape

    for result in results:

        for box in result.boxes:

            x1,y1,x2,y2 = map(int,box.xyxy[0])

            conf = float(box.conf[0])

            if conf < 0.4:
                continue

            area = (x2-x1)*(y2-y1)

            center = (x1+x2)//2

            if area > 50000:

                if center < w//3:

                    command = "MOVE RIGHT"

                elif center > 2*w//3:

                    command = "MOVE LEFT"

                else:

                    command = "STOP"

    commands.append([frame_no,command])

    frame_no += 1

video.release()

df = pd.DataFrame(
    commands,
    columns=["Frame","Control"]
)

df.to_csv(
    "control_log.csv",
    index=False
)

print(df.head())


0: 384x640 2 persons, 122.6ms
Speed: 5.0ms preprocess, 122.6ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 139.6ms
Speed: 3.6ms preprocess, 139.6ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 156.0ms
Speed: 3.7ms preprocess, 156.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 145.8ms
Speed: 4.2ms preprocess, 145.8ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 145.5ms
Speed: 4.5ms preprocess, 145.5ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 136.5ms
Speed: 5.3ms preprocess, 136.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 139.2ms
Speed: 6.3ms preprocess, 139.2ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 138.9ms
Speed: 6.0ms preprocess, 138.9ms inference, 1.1ms postprocess per im

In [8]:
files.download("control_log.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>